# Análise de Resultados — Monitoramento de Riscos Ambientais
## Sistema de Detecção de Incêndios Florestais (Dynamic Programming)

**Stratfy — FIAP Global Solution 2026 (Turma 2ESPH, Engenharia de Software)**

| Integrante | RM |
|---|---|
| Anthony Sforzin | RM562096 |
| Luigi Mendes Cabrini | RM563552 |
| Rogério Cruz Arroyo | RM563517 |
| Bruno Koeke | RM561309 |

Este notebook é **interativo** e reúne a análise dos algoritmos de árvores e grafos aplicados
a dados reais de focos de calor (INPE/BDQueimadas 2025 — 3.466.399 focos agregados em 5.512
municípios). O ponto central é a **escala de decisão** (>= 4 níveis) que orienta *qual algoritmo
usar* conforme o trade-off entre qualidade da solução e custo computacional.

## 1. Preparação do ambiente e carga dos dados reais

In [ ]:
import os, sys, json

# torna os módulos de src/ importáveis a partir de notebooks/
RAIZ = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
SRC = os.path.join(RAIZ, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from loader import carregar_municipios, cenario_matopiba, cenario_amazonia, subgrafo_forca_bruta
from data_structures import BinarySearchTree
from greedy import prim_mst, dijkstra, hub_otimo
from brute_force import todas_arvores_geradoras, todos_os_caminhos
from performance_monitor import rodar_benchmark, imprimir_tabela

municipios = carregar_municipios()
print(f'Municípios carregados (com centroide real): {len(municipios)}')

## 2. Construção dos cenários (dados reais)

- **Cenário B — MATOPIBA**: os 24 municípios com mais focos em MA, TO, PI e BA (fronteira
  agropecuária do Cerrado, onde a pressão de fogo em 2025 foi máxima).
- **Cenário D — Amazônia**: os 24 municípios do bioma Amazônia com mais focos (arco do
  desmatamento — PA, MT, RO, AM).

In [ ]:
g_mato = cenario_matopiba(municipios, n_nos=24, k_vizinhos=3)
g_amaz = cenario_amazonia(municipios, n_nos=24, k_vizinhos=3)

for g in (g_mato, g_amaz):
    print(f'{g.nome}: {g.num_vertices()} vértices | {g.num_arestas()} arestas | conexo={g.eh_conexo()}')

## 3. Árvore Binária de Busca — triagem de risco por intervalo

Inserimos os municípios na BST chaveados pelo **índice de risco** (∈ [0,1]) e usamos a
*busca por intervalo* para isolar rapidamente os municípios em estado crítico.

In [ ]:
bst = BinarySearchTree()
for m in g_mato.vertices.values():
    bst.inserir(m.risco, {'nome': m.nome, 'uf': m.uf, 'focos': m.n_focos})

print(f'BST: n={len(bst)} | altura={bst.altura()}')
print('In-order (crescente por risco) — 5 menores:')
for no in bst.percurso_in_order()[:5]:
    print(f"   {no.chave:.3f}  {no.payload['nome']} ({no.payload['uf']})")

print('\nMunicípios CRÍTICOS (risco ∈ [0.90, 1.00]):')
for no in bst.buscar_intervalo(0.90, 1.00):
    print(f"   {no.chave:.3f}  {no.payload['nome']} ({no.payload['uf']}) — {no.payload['focos']} focos")

## 4. Guloso vs Força Bruta — validação cruzada

Em um subgrafo pequeno (N=10) confirmamos que o **Prim** (guloso) encontra exatamente a
mesma MST que a **enumeração exaustiva** (oráculo), e que o **Dijkstra** coincide com o
melhor caminho enumerado por força bruta.

In [ ]:
sub = subgrafo_forca_bruta(g_mato, n_max=10)
fb = todas_arvores_geradoras(sub)
prim = prim_mst(sub)
print(f'MST força bruta = {fb.custo_total:.4f} h  ({fb.arvores_avaliadas} árvores, {fb.chamadas_recursivas} chamadas)')
print(f'MST Prim        = {prim.custo_total:.4f} h  ({prim.operacoes} operações de heap)')
print('Coincidem (Prim é exato):', abs(fb.custo_total - prim.custo_total) < 1e-6)

ids = sub.ids(); o, d = ids[0], ids[-1]
fbc = todos_os_caminhos(sub, o, d)
dij = dijkstra(sub, o)
print(f'\nCaminho mín {sub.vertices[o].nome}→{sub.vertices[d].nome}:')
print(f'   força bruta = {fbc.melhor_custo:.4f} h | Dijkstra = {dij.distancias[d]:.4f} h')

## 5. Benchmark de desempenho (N = 5, 8, 10, 12, 20, 50, 100)

Medimos tempo (ms), memória (MB) e nº de operações elementares. A força bruta só é viável
até N≈12 — a partir daí o nº de árvores geradoras candidatas cresce de forma combinatória
(teorema de Cayley: $n^{n-2}$ árvores geradoras em um grafo completo de $n$ vértices).

In [ ]:
base = sorted([m for m in municipios if m.uf in {'MA','TO','PI','BA'}], key=lambda m: m.n_focos, reverse=True)
linhas = rodar_benchmark(base, salvar=False)
imprimir_tabela(linhas)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ns_fb = [l.n for l in linhas if l.forca_bruta and l.forca_bruta.viavel]
ops_fb = [l.forca_bruta.operacoes for l in linhas if l.forca_bruta and l.forca_bruta.viavel]
ns_g = [l.n for l in linhas]
ops_g = [l.guloso.operacoes for l in linhas]

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(ns_fb, ops_fb, 'o-', color='#d32f2f', label='Força Bruta (chamadas recursivas)')
ax.plot(ns_g, ops_g, 's-', color='#2e7d32', label='Guloso (operações de heap)')
ax.set_yscale('log'); ax.set_xlabel('N'); ax.set_ylabel('Operações elementares (log)')
ax.set_title('Operações elementares × N'); ax.grid(True, which='both', ls=':', alpha=0.5); ax.legend()
plt.show()

## 6. ESCALA DE DECISÃO (trade-off qualidade × custo computacional)

A escala abaixo orienta a **escolha do algoritmo** segundo o tamanho do problema (N = nº de
municípios a coordenar) e a criticidade da decisão. Combina os dados empíricos do benchmark
com a complexidade teórica.

| Nível | Faixa de N | Algoritmo recomendado | Garantia de qualidade | Custo computacional | Quando usar |
|---|---|---|---|---|---|
| **1 — Exato/Auditoria** | N ≤ 12 | Força Bruta (backtracking) | **Ótimo global** garantido | Alto: até ~6×10⁵ chamadas em N=12 (~0,5 s) | Validar/auditar o guloso; instâncias minúsculas e decisões críticas onde o ótimo precisa ser provado |
| **2 — Guloso exato** | 12 < N ≤ 10³ | Prim (MST) + Dijkstra com heap | **Ótimo** (Prim/Dijkstra são exatos), gap = 0% | Baixo: O(E log V); ms | Planejamento operacional diário de bases e rotas de brigadas — caso de uso padrão |
| **3 — Guloso escalável** | 10³ < N ≤ 10⁵ | Prim/Dijkstra com lista de adjacência + heap | Ótimo, mas exige grafo esparso (k-vizinhos) | Médio: depende de manter E = O(k·V) | Cobertura estadual/regional (centenas a milhares de municípios) com grafo de proximidade podado |
| **4 — Heurístico/Aproximado** | N > 10⁵ | Heurísticas (clustering espacial + Prim local, A*) | Sub-ótimo controlado (gap aceitável) | Limitado por tempo real | Cobertura nacional em tempo real; aceita-se perder otimalidade por resposta imediata |

**Leitura do trade-off:** subir um nível na escala troca *garantia de otimalidade* por
*escalabilidade*. Como Prim e Dijkstra são **exatos** e custam apenas O(E log V), o nível 2/3
domina a força bruta em todos os critérios para N > 12 — por isso o guloso é a escolha
operacional padrão, reservando a força bruta apenas para auditoria (nível 1).

In [ ]:
def classificar_nivel(n):
    if n <= 12:      return (1, 'Exato/Auditoria', 'Força Bruta (backtracking)')
    if n <= 1000:    return (2, 'Guloso exato', 'Prim + Dijkstra (heap)')
    if n <= 100000:  return (3, 'Guloso escalável', 'Prim/Dijkstra esparso (k-vizinhos)')
    return (4, 'Heurístico/Aproximado', 'Clustering + Prim local / A*')

for n in [8, 12, 24, 500, 5512, 200000]:
    nivel, rotulo, alg = classificar_nivel(n)
    print(f'N={n:>7}  →  Nível {nivel} ({rotulo}): {alg}')

## 7. Conclusão e conexão com os ODS

- A **BST** entrega triagem de risco em O(log n + k) por consulta de intervalo — útil para
  priorizar municípios críticos em tempo de resposta.
- O **Prim** define a malha mínima de deslocamento entre bases (menor tempo total de
  patrulha) e o **Dijkstra** a rota mais rápida a partir de um hub — ambos **exatos** e baratos.
- A **força bruta** comprova a otimalidade do guloso (gap = 0%) e evidencia, por contraste, a
  explosão combinatória que torna a enumeração inviável acima de N≈12.

**ODS atendidos:** ODS 2 (segurança alimentar — proteção da fronteira agropecuária do
MATOPIBA), ODS 9 (infraestrutura/inovação — sistema de apoio à decisão), ODS 11 (cidades e
comunidades resilientes — proteção de municípios), ODS 13 (ação climática — resposta a focos
de calor e mitigação de emissões por queimadas).